# Algorithm 1 — Composite Session Quality Engine

**File:** `src/CopilotScope.Collector/Quality/QualityEngine.cs`

Produces a **0–100 composite quality score** with a confidence value by aggregating up to six
independent components.  Only components that have real data contribute; missing components do not
pollute the score with a neutral prior (v2 design).

---

## 1  Component Definitions

| Component | Base weight $w_i$ | Sub-formula |
|---|---|---|
| Reliability | 0.25 | $r^2$ where $r = 1 - \frac{2 e_{\text{LLM}} + e_{\text{tool}}}{2 c_{\text{LLM}} + c_{\text{tool}}}$ |
| Acceptance | 0.20 | $0.6 \cdot \alpha + 0.4 \cdot \bar{s}$ |
| Friction | 0.20 | $\frac{1}{N}\sum_{i=1}^{N} \tau_i$ (TFRA mean, see notebook 02) |
| Latency | 0.15 | $L(p_{50})$ (log-linear curve, see notebook 05) |
| Feedback | 0.10 | $\frac{u^+}{u^+ + u^-}$ |
| Efficiency | 0.10 | $\frac{1}{|P|}\sum_p \text{eff}_p$ |

where $\alpha$ = acceptance ratio, $\bar{s}$ = mean survival score, $u^+$/$u^-$ = thumbs up/down.

---

## 2  Composite Score with Weight Renormalisation

Let $\mathcal{I} \subseteq \{1,\dots,6\}$ be the set of **informative** components (those with at
least one sample).  Define coverage $C = \sum_{i \in \mathcal{I}} w_i \leq 1$.

$$
\boxed{
  Q = \frac{100}{C} \sum_{i \in \mathcal{I}} w_i \cdot v_i
}
$$

When $\mathcal{I} = \varnothing$ the system returns the neutral prior $Q = 70$.

---

## 3  Confidence

Each component $i$ contributes a **sample ramp** $\rho_i = \min(1,\, n_i / 5)$ that reaches 1.0
after five data points.  The weighted ramp over informative components is:

$$
\rho = \frac{\sum_{i \in \mathcal{I}} w_i \cdot \rho_i}{C}
$$

$$
\boxed{\text{Confidence} = C \cdot \rho}
$$

A confidence of 1.0 means full weight coverage AND at least 5 samples per component.

---

## 4  Relative Score (z-score and percentile rank)

Given history window $\{Q_1, \dots, Q_H\}$ with mean $\mu$ and standard deviation $\sigma$:

$$
z = \frac{Q - \mu}{\sigma}, \qquad
\text{PercentileRank} = \frac{|\{j : Q_j \leq Q\}|}{H}
$$

Grade thresholds: **excellent** $\geq 85$, **good** $\geq 70$, **fair** $\geq 55$, **poor** $\geq 40$, **critical** $< 40$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

PRIOR = 0.70

@dataclass
class Component:
    name: str
    weight: float
    value: float
    samples: int
    detail: str = ''

def composite_score(components: list[Component]):
    informative = [c for c in components if c.samples > 0]
    if not informative:
        return PRIOR * 100, 0.0
    coverage = sum(c.weight for c in informative)
    score = sum(c.weight * c.value for c in informative) / coverage * 100.0
    ramp = sum(c.weight * min(1.0, c.samples / 5.0) for c in informative) / coverage
    confidence = coverage * ramp
    return round(score, 1), round(confidence, 2)

def grade(score: float) -> str:
    if score >= 85: return 'excellent'
    if score >= 70: return 'good'
    if score >= 55: return 'fair'
    if score >= 40: return 'poor'
    return 'critical'

# --- Reliability sub-formula ---
def reliability(chat_errors, chat_calls, tool_errors, tool_calls):
    calls = chat_calls + tool_calls
    if calls == 0:
        return PRIOR
    weighted_errors = chat_errors * 2.0 + tool_errors
    weighted_calls  = chat_calls  * 2.0 + tool_calls
    error_free = np.clip(1.0 - weighted_errors / max(1, weighted_calls), 0, 1)
    return error_free ** 2   # quadratic penalty

print('=== Unit result table ===')
print(f'{"Scenario":<35} {"Score":>6} {"Conf":>6} {"Grade":<10}')
print('-' * 65)

scenarios = [
    ('Perfect session (all data)',
     [Component('reliability', 0.25, reliability(0, 20, 0, 15), 35),
      Component('acceptance',  0.20, 0.6*0.95 + 0.4*0.90, 40),
      Component('friction',    0.20, 0.97, 10),
      Component('latency',     0.15, 0.92, 10),
      Component('feedback',    0.10, 1.00,  5),
      Component('efficiency',  0.10, 0.85, 30)]),
    ('Two LLM errors, no other data',
     [Component('reliability', 0.25, reliability(2, 10, 0, 0), 10),
      Component('acceptance',  0.20, PRIOR,  0),
      Component('friction',    0.20, PRIOR,  0),
      Component('latency',     0.15, PRIOR,  0),
      Component('feedback',    0.10, PRIOR,  0),
      Component('efficiency',  0.10, PRIOR,  0)]),
    ('No data at all (prior)',
     [Component('reliability', 0.25, PRIOR, 0),
      Component('acceptance',  0.20, PRIOR, 0),
      Component('friction',    0.20, PRIOR, 0),
      Component('latency',     0.15, PRIOR, 0),
      Component('feedback',    0.10, PRIOR, 0),
      Component('efficiency',  0.10, PRIOR, 0)]),
    ('High rejection rate, good latency',
     [Component('reliability', 0.25, reliability(0, 15, 1, 8), 23),
      Component('acceptance',  0.20, 0.6*0.30 + 0.4*0.55, 20),
      Component('friction',    0.20, 0.80, 8),
      Component('latency',     0.15, 0.95, 8),
      Component('feedback',    0.10, PRIOR,  0),
      Component('efficiency',  0.10, PRIOR,  0)]),
]

for name, comps in scenarios:
    s, c = composite_score(comps)
    print(f'{name:<35} {s:>6.1f} {c:>6.2f} {grade(s):<10}')

In [ ]:
# --- Reliability quadratic penalty visualisation ---
error_rates = np.linspace(0, 1, 200)
rel_scores  = np.clip(1 - error_rates, 0, 1) ** 2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(error_rates * 100, rel_scores, lw=2, color='steelblue')
axes[0].set_xlabel('Weighted error rate (%)')
axes[0].set_ylabel('Reliability value')
axes[0].set_title('Reliability — quadratic penalty')
axes[0].grid(alpha=0.3)
axes[0].annotate('$(1 - e)^2$', xy=(50, 0.25), fontsize=14, color='steelblue')

# Confidence vs. samples (single component illustration)
n_samples = np.arange(0, 15)
ramp = np.minimum(1.0, n_samples / 5.0)
axes[1].plot(n_samples, ramp, lw=2, color='darkorange')
axes[1].axvline(5, ls='--', color='gray', alpha=0.6, label='full ramp at n=5')
axes[1].set_xlabel('Samples per component')
axes[1].set_ylabel('Sample ramp $\\rho_i$')
axes[1].set_title('Confidence ramp')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quality_engine_plots.png', dpi=150)
plt.show()
print('Saved quality_engine_plots.png')